# Vectyfi Radar — V3 Ultimate Honest Baseline

### Mission: VEC-52

Build the **ultimate leak-free baseline** on the `export_CFC_2018_2023.csv` dataset.

**Key upgrades over V2:**
1. **Advanced Feature Engineering** — Temporal signals (`month`, `year`) extracted from `DT_DISPATCH`, `DURATION` as a numeric feature with median imputation, plus additional structural CN columns (`CAE_TYPE`, `B_EU_FUNDS`, `B_GPA`, `LOTS_NUMBER`).
2. **Low-Frequency Grouping** — CPV codes and country codes with fewer than 1% representation are grouped into `Other` to reduce dimensionality noise.
3. **Class Imbalance Calibration** — The exact `scale_pos_weight` ratio is computed and injected into XGBoost to heavily penalize errors on the rare "Not Awarded" class.
4. **Precision-Recall AUC** — For highly imbalanced datasets, PR-AUC is the true quality metric. We now report and plot both ROC-AUC and PR-AUC.
5. **Zero Target Leakage** — The same 21-column CAN exorcism from V2 is enforced with surgical precision.

In [ ]:
# ============================================================
# Cell 1: Imports & Environment
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score,
    roc_curve
)
import warnings
warnings.filterwarnings('ignore')

# Dark theme with teal/emerald accent palette
plt.style.use('dark_background')
TEAL   = '#00d1b2'
CORAL  = '#ff6b6b'
GOLD   = '#f7d354'
SLATE  = '#1a1a2e'

print('\u2705 Libraries loaded. V3 Ultimate Honest Baseline ready.')

In [ ]:
# ============================================================
# Cell 2: Data Loading & Strict Deduplication
# ============================================================
file_path = '/Users/edu/Edu/testproject/tenderpilot_data/data/export_CFC_2018_2023.csv'

# V3 loads significantly more columns than V2 for advanced feature engineering.
# We deliberately load target-link columns too (to build y), then surgically drop them.
USECOLS = [
    # Identity
    'ID_NOTICE_CN',
    # Core CN features (leak-free)
    'CPV', 'ISO_COUNTRY_CODE', 'B_FRA_AGREEMENT', 'TYPE_OF_CONTRACT',
    'CAE_TYPE', 'B_EU_FUNDS', 'B_GPA', 'LOTS_NUMBER',
    'B_DYN_PURCH_SYST', 'B_ELECTRONIC_AUCTION', 'B_RENEWALS', 'B_OPTIONS',
    'TOP_TYPE', 'MAIN_ACTIVITY',
    # Temporal (for month/year extraction)
    'DT_DISPATCH',
    # Numeric (leak-free pre-award estimated value)
    'VALUE_EURO', 'DURATION',
    # Target columns (used to build y, then dropped)
    'WIN_NAME', 'AWARD_VALUE_EURO',
    # Also load FUTURE_CAN_ID as a fallback target indicator
    'FUTURE_CAN_ID'
]

print(f'Loading V3 feature set from {file_path}...')

# Safely load only columns that actually exist in the file
headers = pd.read_csv(file_path, nrows=0).columns.tolist()
available_cols = [c for c in USECOLS if c in headers]
missing_cols = [c for c in USECOLS if c not in headers]
if missing_cols:
    print(f'\u26a0\ufe0f  Columns not found in CSV (skipped): {missing_cols}')

df_raw = pd.read_csv(file_path, usecols=available_cols, low_memory=False)
print(f'Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns.')

# Strict deduplication on the Contract Notice ID
df_dedup = df_raw.drop_duplicates(subset=['ID_NOTICE_CN'], keep='first').copy()
print(f'After dedup: {df_dedup.shape[0]:,} unique notices.')

# Sample 500k for memory-safe training
if len(df_dedup) > 500_000:
    df = df_dedup.sample(n=500_000, random_state=42).copy()
else:
    df = df_dedup.copy()

print(f'\u2705 V3 Dataset prepared: {df.shape[0]:,} rows ready for target engineering.')

In [ ]:
# ============================================================
# Cell 3: Target Creation & The Leakage Exorcism
# ============================================================
print('Engineering target variable y...')

# A notice resulted in an award if WIN_NAME is populated OR AWARD_VALUE_EURO > 0
if 'WIN_NAME' in df.columns and 'AWARD_VALUE_EURO' in df.columns:
    df['y'] = ((df['WIN_NAME'].notna()) | (df['AWARD_VALUE_EURO'] > 0)).astype(int)
elif 'FUTURE_CAN_ID' in df.columns:
    df['y'] = df['FUTURE_CAN_ID'].notna().astype(int)
else:
    raise ValueError('No valid target columns found in dataset!')

# --- Target Distribution ---
n_pos = df['y'].sum()
n_neg = len(df) - n_pos
imbalance_ratio = n_neg / n_pos

print(f'\n{"="*50}')
print(f'  Awarded (y=1):     {n_pos:>8,}  ({n_pos/len(df)*100:.1f}%)')
print(f'  Not Awarded (y=0): {n_neg:>8,}  ({n_neg/len(df)*100:.1f}%)')
print(f'  Imbalance Ratio:   1 : {imbalance_ratio:.4f}')
print(f'{"="*50}')

# --- The Leakage Exorcism ---
# These 21 CAN/Result columns reveal the outcome. They MUST be purged.
CAN_LEAK_COLUMNS = [
    'WIN_NAME', 'WIN_NATIONALID', 'WIN_ADDRESS', 'WIN_TOWN', 'WIN_POSTAL_CODE',
    'WIN_COUNTRY_CODE', 'B_AWARDED_TO_A_GROUP', 'B_CONTRACTOR_SME', 'NUMBER_OFFERS',
    'NUMBER_TENDERS_SME', 'NUMBER_TENDERS_OTHER_EU', 'NUMBER_TENDERS_NON_EU',
    'NUMBER_OFFERS_ELECTR', 'AWARD_VALUE_EURO', 'AWARD_VALUE_EURO_FIN_1',
    'AWARD_EST_VALUE_EURO', 'DT_AWARD', 'ID_AWARD', 'B_SUBCONTRACTED',
    'INFO_ON_NON_AWARD', 'CONTRACT_NUMBER', 'FUTURE_CAN_ID', 'FUTURE_CAN_ID_ESTIMATED'
]

cols_to_drop = set(CAN_LEAK_COLUMNS) | {'ID_NOTICE_CN', 'y'}
existing_drops = [c for c in cols_to_drop if c in df.columns]

X_raw = df.drop(columns=existing_drops).copy()
y     = df['y'].copy()

print(f'\nLeakage Exorcism: Dropped {len(existing_drops)} columns.')
print(f'Remaining feature columns: {list(X_raw.columns)}')
print('\u2705 Zero target leakage guaranteed.')

In [ ]:
# ============================================================
# Cell 4: Advanced Feature Engineering (V3 NEW)
# ============================================================
print('Building V3 advanced feature matrix...\n')
X = pd.DataFrame(index=X_raw.index)

# ---- 1. Temporal Features from DT_DISPATCH ----
if 'DT_DISPATCH' in X_raw.columns:
    dt = pd.to_datetime(X_raw['DT_DISPATCH'], errors='coerce')
    X['dispatch_month'] = dt.dt.month.fillna(0).astype(int)
    X['dispatch_year']  = dt.dt.year.fillna(0).astype(int)
    X['dispatch_quarter'] = dt.dt.quarter.fillna(0).astype(int)
    print(f'  \u2713 Temporal features extracted (month, year, quarter) from DT_DISPATCH')

# ---- 2. DURATION as numeric (median imputation) ----
if 'DURATION' in X_raw.columns:
    X['duration'] = pd.to_numeric(X_raw['DURATION'], errors='coerce')
    median_dur = X['duration'].median()
    X['duration'] = X['duration'].fillna(median_dur)
    print(f'  \u2713 DURATION -> numeric float (median imputed: {median_dur:.1f})')

# ---- 3. VALUE_EURO (pre-award estimated value, NOT the final award value) ----
if 'VALUE_EURO' in X_raw.columns:
    X['value_euro_log'] = np.log1p(pd.to_numeric(X_raw['VALUE_EURO'], errors='coerce').fillna(0).clip(lower=0))
    print(f'  \u2713 VALUE_EURO -> log1p transformed')

# ---- 4. LOTS_NUMBER as numeric ----
if 'LOTS_NUMBER' in X_raw.columns:
    X['lots_number'] = pd.to_numeric(X_raw['LOTS_NUMBER'], errors='coerce').fillna(1).astype(int)
    print(f'  \u2713 LOTS_NUMBER -> numeric integer')

# ---- 5. Binary flags (already 0/1 or Y/N) ----
binary_cols = {
    'B_FRA_AGREEMENT': 'is_framework',
    'B_EU_FUNDS': 'is_eu_funded',
    'B_GPA': 'is_gpa',
    'B_DYN_PURCH_SYST': 'is_dps',
    'B_ELECTRONIC_AUCTION': 'is_e_auction',
    'B_RENEWALS': 'is_renewable',
    'B_OPTIONS': 'has_options',
}
for src, dst in binary_cols.items():
    if src in X_raw.columns:
        X[dst] = X_raw[src].map({'Y': 1, 'N': 0, 1: 1, 0: 0}).fillna(0).astype(int)
print(f'  \u2713 {len(binary_cols)} binary flag features mapped')

# ---- 6. CPV Group (first 2 digits) with low-frequency grouping ----
if 'CPV' in X_raw.columns:
    cpv_raw = X_raw['CPV'].astype(str).str[:2].replace('na', 'Other').fillna('Other')
    # Group categories that appear in < 1% of rows into 'Other'
    cpv_counts = cpv_raw.value_counts(normalize=True)
    rare_cpvs = cpv_counts[cpv_counts < 0.01].index
    cpv_raw = cpv_raw.replace(rare_cpvs, 'Other')
    X['cpv_group'] = cpv_raw
    print(f'  \u2713 CPV -> {cpv_raw.nunique()} groups (low-frequency merged into Other)')

# ---- 7. Country Code with low-frequency grouping ----
if 'ISO_COUNTRY_CODE' in X_raw.columns:
    cc_raw = X_raw['ISO_COUNTRY_CODE'].fillna('Other')
    cc_counts = cc_raw.value_counts(normalize=True)
    rare_cc = cc_counts[cc_counts < 0.01].index
    cc_raw = cc_raw.replace(rare_cc, 'Other')
    X['country'] = cc_raw
    print(f'  \u2713 ISO_COUNTRY -> {cc_raw.nunique()} groups (low-frequency merged into Other)')

# ---- 8. Contract Type & CAE Type ----
if 'TYPE_OF_CONTRACT' in X_raw.columns:
    X['contract_type'] = X_raw['TYPE_OF_CONTRACT'].fillna('Unknown')
if 'CAE_TYPE' in X_raw.columns:
    X['cae_type'] = X_raw['CAE_TYPE'].fillna('Unknown')
if 'TOP_TYPE' in X_raw.columns:
    X['top_type'] = X_raw['TOP_TYPE'].fillna('Unknown')
if 'MAIN_ACTIVITY' in X_raw.columns:
    X['main_activity'] = X_raw['MAIN_ACTIVITY'].fillna('Unknown')
    # Group low-frequency activities
    act_counts = X['main_activity'].value_counts(normalize=True)
    rare_act = act_counts[act_counts < 0.01].index
    X['main_activity'] = X['main_activity'].replace(rare_act, 'Other')

print(f'\nPre-encoding feature count: {X.shape[1]}')

# ---- 9. One-Hot Encoding ----
cat_cols = [c for c in ['cpv_group', 'country', 'contract_type', 'cae_type', 'top_type', 'main_activity']
            if c in X.columns]

print(f'Applying One-Hot Encoding to: {cat_cols}')
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)

print(f'\n\u2705 Final V3 feature matrix: {X_encoded.shape[0]:,} rows x {X_encoded.shape[1]} features')

In [ ]:
# ============================================================
# Cell 5: Algorithmic Calibration — The Imbalance Fix (V3 NEW)
# ============================================================
print('Calibrating for class imbalance...\n')

# Compute the exact balancing weight
# scale_pos_weight = count(Negative) / count(Positive)
# This tells XGBoost to treat each "Not Awarded" sample as N times more important
scale_weight = n_neg / n_pos
print(f'  Class 0 (Not Awarded): {n_neg:,}')
print(f'  Class 1 (Awarded):     {n_pos:,}')
print(f'  scale_pos_weight:      {scale_weight:.4f}')
print(f'\n  \u2192 XGBoost will penalize misclassifications on the minority class')
print(f'    {1/scale_weight:.2f}x more aggressively than the majority class.')

# ---- Train / Test Split (80/20, stratified) ----
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\n  Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

# ---- XGBoost V3 with scale_pos_weight ----
print(f'\nTraining XGBoost V3 on {X_train.shape[0]:,} samples with {X_train.shape[1]} features...')

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print('\n\u2705 V3 Model training complete.')

In [ ]:
# ============================================================
# Cell 6: Advanced Evaluation & Metrics
# ============================================================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ---- 1. ROC-AUC ----
roc_auc = roc_auc_score(y_test, y_prob)

# ---- 2. PR-AUC (the true metric for imbalanced data) ----
pr_auc = average_precision_score(y_test, y_prob)

print(f'\n{"="*60}')
print(f'  \U0001f3c6 V3 ULTIMATE HONEST BASELINE — RESULTS')
print(f'{"="*60}')
print(f'  ROC-AUC Score:             {roc_auc:.4f}')
print(f'  Precision-Recall AUC:      {pr_auc:.4f}   \u2190 (The True Metric)')
print(f'{"="*60}')

# ---- 3. Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred)
print(f'\n--- Confusion Matrix ---')
print(f'               Predicted: No   Predicted: Yes')
print(f'  Actual: No     {cm[0][0]:>7,}        {cm[0][1]:>7,}')
print(f'  Actual: Yes    {cm[1][0]:>7,}        {cm[1][1]:>7,}')

# ---- 4. Classification Report ----
print(f'\n--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=['Not Awarded', 'Awarded']))
print('\u2705 Evaluation complete. Generating visualizations...')

In [ ]:
# ============================================================
# Cell 7: Visualization Dashboard (Dark Theme + Teal/Emerald)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.patch.set_facecolor(SLATE)
for ax in axes:
    ax.set_facecolor(SLATE)

# ---- Plot 1: ROC Curve ----
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, color=TEAL, linewidth=2.5, label=f'V3 ROC-AUC = {roc_auc:.4f}')
axes[0].plot([0, 1], [0, 1], color='grey', linestyle='--', linewidth=1, alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.08, color=TEAL)
axes[0].set_xlabel('False Positive Rate', color='white', fontsize=11)
axes[0].set_ylabel('True Positive Rate', color='white', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=14, fontweight='bold', color='white')
axes[0].legend(loc='lower right', fontsize=10)

# ---- Plot 2: Precision-Recall Curve ----
precision, recall, _ = precision_recall_curve(y_test, y_prob)
axes[1].plot(recall, precision, color=CORAL, linewidth=2.5, label=f'V3 PR-AUC = {pr_auc:.4f}')
axes[1].fill_between(recall, precision, alpha=0.08, color=CORAL)
# Baseline = proportion of positives
baseline_pr = n_pos / (n_pos + n_neg)
axes[1].axhline(y=baseline_pr, color='grey', linestyle='--', linewidth=1, alpha=0.5, label=f'Baseline ({baseline_pr:.2f})')
axes[1].set_xlabel('Recall', color='white', fontsize=11)
axes[1].set_ylabel('Precision', color='white', fontsize=11)
axes[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold', color='white')
axes[1].legend(loc='upper right', fontsize=10)

# ---- Plot 3: Feature Importance (Top 15) ----
importances = pd.Series(model.feature_importances_, index=X_train.columns)
top_15 = importances.sort_values(ascending=True).tail(15)
top_15.plot(kind='barh', ax=axes[2], color=TEAL, edgecolor=SLATE)
axes[2].set_title('Top 15 Feature Importances', fontsize=14, fontweight='bold', color='white')
axes[2].set_xlabel('Relative Importance (Gain)', color='white', fontsize=11)
axes[2].set_ylabel('')

fig.suptitle('Vectyfi Radar — V3 Ultimate Honest Baseline Dashboard',
             fontsize=18, fontweight='bold', color=GOLD, y=1.02)
plt.tight_layout()
plt.show()

print('\n\u2705 V3 Dashboard rendered successfully.')

---

## V3 Summary

| Metric | V2 Honest Baseline | V3 Ultimate Baseline |
|---|---|---|
| Features | 4 (CPV, Country, FRA, Contract Type) | 15+ structural + temporal + numeric |
| Dimensionality (post-OHE) | ~79 columns | Significantly reduced via low-freq grouping |
| Class Imbalance Fix | None (`scale_pos_weight=1`) | `scale_pos_weight` auto-calculated |
| ROC-AUC | 0.6793 | *(run to see)* |
| PR-AUC | Not measured | *(run to see)* |
| Target Leakage | Zero | Zero |

**Next Step:** V4 will introduce **Unsupervised Micro-Market Clustering** (K-Means/HDBSCAN on MiniLM-L6-v2 embeddings) and the **LLM Sniper Feature Extraction** (Ollama + Gemma) to push beyond the tabular ceiling.